# 한계효용 이론 검증

**검증할 가설**

> 다음 정보 1단위의 한계효용(MU = p(1−p)의 기대 감소)이 비용보다 클 때만 MU가 가장 큰 것부터 산다.
> 그러면 총효용은 따로 관리하지 않아도 (a) 플랜 A보다 항상 크고, (b) 시간이 갈수록 커진다.

**실험 구성** (프레임워크의 `extraction` / `online` 구조를 최소한으로 재현)

- 사건 1,500개 (실제 경기 1,517개와 같은 규모). 사건마다 결과 y ∈ {0, 1}.
- **플랜 A**: 정보를 사지 않는 기본 모델. 참 확률에 잡음이 섞인 확률 p0.
- **채널 40개**: 각 채널은 정확도 a(0.48~0.64)로 y를 맞히는 신호를 냄. 하나 살 때마다 비용 0.003.
- **파이프라인**: p0에서 시작해 MU가 가장 큰 채널을 MU > 비용인 동안 하나씩 사고 베이즈 갱신.
  결과가 확정되면 사용한 채널의 정확도 기록을 갱신.
- **측정**: 사건마다 Δ = Brier(플랜 A) − [Brier(파이프라인) + 비용]. Δ > 0이면 파이프라인이 이득.
  **예상 이득**(구매 시점에 계산한 MU − 비용의 합)도 같이 기록해, 실현 이득과 비교함.

**시나리오**: 앞 대화에서 짚은 전제 위반 조건(승자의 저주, 베낀 정보, 환경 변화)을 하나씩 켬.

## 1. 시뮬레이터

In [ ]:
import numpy as np

def logit(p): return np.log(p / (1 - p))
def sig(x): return 1 / (1 + np.exp(-x))

def exp_pq_after(p, a):
    """정확도 a인 신호 하나를 본 뒤의 기대 p'(1-p') (신호 독립 가정)."""
    ps1 = p * a + (1 - p) * (1 - a)
    p1 = p * a / ps1
    p0 = p * (1 - a) / (1 - ps1)
    return ps1 * p1 * (1 - p1) + (1 - ps1) * p0 * (1 - p0)

def run(seed, estimator="naive", copy=False, drift=False,
        T=1500, K=40, cost=0.003, n0=10, planA_noise=0.4,
        prior_n=20, z=1.0, forget=1.0):
    """
    사건 T개를 순서대로 처리. 사건마다 플랜 A 확률 p0에서 시작해
    한계효용(MU = p(1-p)의 기대 감소)이 가장 큰 채널을 MU > 비용인 동안 하나씩 산다.
    반환: 사건별 실현 이득 Δ = Brier(플랜A) - [Brier(파이프라인) + 비용],
          사건별 예상 이득 (구매 시점에 계산한 MU 합 - 비용)
    """
    rng = np.random.default_rng(seed)
    a_true = rng.uniform(0.48, 0.64, K)          # 채널의 실제 정확도
    src = np.arange(K)                           # 신호 출처 (베끼면 다른 채널)
    if copy:                                     # 절반 채널이 상위 5개를 베낌
        top = np.argsort(-a_true)[:5]
        for k in rng.choice(K, K // 2, replace=False):
            src[k] = rng.choice(top)
    acc = a_true.copy()                          # 원신호 정확도 (drift로 변함)
    win = rng.binomial(n0, acc[src]).astype(float)  # 초기 이력 n0건
    n = np.full(K, float(n0))
    realized, predicted = np.zeros(T), np.zeros(T)

    for t in range(T):
        if drift and t == T // 2:                # 중간에 좋던 채널들이 무력화
            acc = np.where(acc > 0.56, 0.47, acc)
        q = rng.beta(2, 2)
        y = bool(rng.random() < q)
        p0 = float(sig(logit(np.clip(q, .02, .98)) + rng.normal(0, planA_noise)))
        raw = np.where(rng.random(K) < acc, y, not y)
        s_k = raw[src]

        if estimator == "oracle":                # 참 정확도를 안다고 가정
            m, slack = acc[src], 0.0
        elif estimator == "shrink":              # 축소 + 불확실성만큼 깎기
            m = (win + 0.5 * prior_n) / (n + prior_n)
            slack = z * np.sqrt(m * (1 - m) / (n + prior_n))
        else:                                    # 관측 비율 그대로
            m, slack = win / n, 0.0
        strength = np.clip(np.abs(m - 0.5) - slack, 0, 0.45)
        a_use = 0.5 + strength                   # 신호를 쓸 때 가정하는 정확도
        flip = m < 0.5                           # 반대로 맞히는 채널은 뒤집어 읽음

        p, spent, gain_pred = p0, 0.0, 0.0
        used = np.zeros(K, bool)
        while True:
            mu = np.where(used, -1.0, p * (1 - p) - exp_pq_after(p, a_use))
            k = int(np.argmax(mu))
            if mu[k] <= cost:
                break
            used[k] = True
            spent += cost
            gain_pred += mu[k] - cost
            lr = np.log(a_use[k] / (1 - a_use[k]))
            p = float(np.clip(sig(logit(p) + (lr if (s_k[k] ^ flip[k]) else -lr)), 1e-4, 1 - 1e-4))

        realized[t] = (p0 - y) ** 2 - ((p - y) ** 2 + spent)
        predicted[t] = gain_pred
        win *= forget                            # 망각 계수 (1.0 = 안 잊음)
        n *= forget
        win[used] += (s_k[used] == y)            # 결과 확정 후 사용한 채널만 갱신
        n[used] += 1
    return realized, predicted

def summarize(name, seeds=60, blocks=10, **kw):
    R, P = zip(*(run(s, **kw) for s in range(seeds)))
    R, P = np.array(R), np.array(P)
    T = R.shape[1]
    B = R.reshape(seeds, blocks, T // blocks).mean(2)
    first, second = R[:, :T // 2].mean(1), R[:, T // 2:].mean(1)
    return dict(
        시나리오=name,
        전체_실현이득=round(R.mean() * 1000, 2),
        예상이득=round(P.mean() * 1000, 2),
        과대평가비율=round(P.mean() / R.mean(), 2) if R.mean() > 0 else float("inf"),
        음수구간_비율=float(np.mean(B.mean(0) < 0)),
        후반이_전반보다_큰_시드=float(np.mean(second > first)),
        손해본_시드=float(np.mean(R.mean(1) < 0)),
        구간별=np.round(B.mean(0) * 1000, 2).tolist(),
    )

SCENARIOS = {
    "1 이상적 (참 정확도 앎)":       dict(estimator="oracle"),
    "2 추정 (표본 10건)":            dict(estimator="naive"),
    "3 추정 + 베끼는 채널":          dict(estimator="naive", copy=True),
    "4 추정 + 중간 변화":            dict(estimator="naive", drift=True),
    "5 축소 추정":                   dict(estimator="shrink"),
    "6 축소 + 베끼는 채널":          dict(estimator="shrink", copy=True),
    "7 축소 + 중간 변화":            dict(estimator="shrink", drift=True),
    "8 약한 축소":                   dict(estimator="shrink", prior_n=5, z=0.3),
    "9 약한 축소 + 중간 변화":       dict(estimator="shrink", prior_n=5, z=0.3, drift=True),
    "10 약한 축소 + 변화 + 망각":    dict(estimator="shrink", prior_n=5, z=0.3, drift=True, forget=0.99),
}

## 2. 실행
시드 60개 기준, 코랩 CPU에서 몇 분 걸립니다. 빨리 보려면 `SEEDS = 20`으로 줄이세요.

In [ ]:
import pandas as pd
from concurrent.futures import ProcessPoolExecutor

SEEDS = 60
with ProcessPoolExecutor() as ex:
    futs = {k: ex.submit(summarize, k, SEEDS, **v) for k, v in SCENARIOS.items()}
    results = [f.result() for f in futs.values()]

df = pd.DataFrame(results).set_index("시나리오")
df.drop(columns="구간별")

**열 읽는 법** (이득 단위: Brier × 1000, 사건 1개당)

| 열 | 뜻 | 가설이 맞다면 |
|---|---|---|
| 전체_실현이득 | 플랜 A 대비 실제 이득 평균 | > 0 |
| 예상이득 | 구매할 때 MU로 계산한 이득 | ≈ 실현이득 |
| 과대평가비율 | 예상 ÷ 실현. 1보다 크면 승자의 저주 | ≈ 1 |
| 음수구간_비율 | 시간 구간 10개 중 플랜 A보다 손해 본 구간 비율 | 0 |
| 후반이_전반보다_큰_시드 | 후반 750건의 이득이 전반보다 큰 시드 비율 | ≈ 1 ("시간이 갈수록 증가") |
| 손해본_시드 | 전체 이득이 음수인 시드 비율 | 0 |

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["axes.unicode_minus"] = False
try:  # 코랩 한글 폰트 (없으면 영어 라벨로 대체)
    import subprocess
    subprocess.run("apt-get -qq install -y fonts-nanum > /dev/null", shell=True)
    import matplotlib.font_manager as fm
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    plt.rcParams["font.family"] = "NanumGothic"
except Exception:
    pass

fig, ax = plt.subplots(figsize=(10, 5))
for r in results:
    ax.plot(range(1, 11), r["구간별"], marker="o", label=r["시나리오"])
ax.axhline(0, color="k", lw=1)
ax.set_xlabel("시간 구간 (사건 150개씩)")
ax.set_ylabel("플랜 A 대비 이득 (Brier ×1000)")
ax.set_title("시간에 따른 순이득: 0 아래면 플랜 A보다 손해")
ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout(); plt.show()

## 3. 사전 실행 결과 (시드 60개, 8~10번은 40개)

| 시나리오 | 실현 이득 | 예상 이득 | 과대평가 | 손해 구간 | 후반 > 전반 | 손해 본 시드 |
|---|---|---|---|---|---|---|
| 1 이상적 (참 정확도 앎) | **+55.2** | 52.9 | 0.96 | 0% | 43% | 0% |
| 2 추정 (표본 10건) | +21.6 | 38.1 | **1.76** | 10% (첫 구간) | 85% | 2% |
| 3 추정 + 베끼는 채널 | +23.0 | 48.1 | **2.09** | 10% (첫 구간) | 98% | 2% |
| 4 추정 + 중간 변화 | **−8.3** | 28.6 | ∞ | **60%** | 3% | **100%** |
| 5 축소 추정 (강함) | +0.3 | 0.2 | 0.79 | 0% | 10% | 8% |
| 6 축소 + 베끼는 채널 | +0.5 | 0.5 | 1.05 | 0% | 15% | 15% |
| 7 축소 + 중간 변화 | +0.0 | 0.2 | 5.07 | 50% | 8% | 8% |
| 8 약한 축소 | +12.3 | 16.4 | 1.33 | 0% | 70% | 5% |
| 9 약한 축소 + 중간 변화 | −2.1 | 11.9 | ∞ | 50% | 5% | 93% |
| 10 약한 축소 + 변화 + 망각 | +1.7 | 4.3 | 2.47 | 10% | 45% | 45% |

**가설별 판정**

- **(a) "플랜 A보다 항상 크다"**: 정확도를 정확히 알 때(1번)만 성립합니다.
  추정치를 쓰면 초반에 손해를 봅니다(2, 3번 첫 구간 −13, −26).
  환경이 바뀌면 시드 100%가 전체적으로 손해입니다(4번).
- **(b) "시간이 갈수록 증가한다"**: 1번에서는 이득이 처음부터 일정하고 더 늘지 않습니다(후반 > 전반 43%, 우연 수준).
  2, 3번의 증가는 초반 손실이 회복되는 모습이지 계속 커지는 것이 아닙니다. 4번에서는 변화 후 계속 감소합니다.
- **승자의 저주**: MU 최댓값을 고르면 예상 이득이 실현 이득의 1.8~2.1배로 부풀려집니다.
  베끼는 채널이 있으면 더 커집니다(2.09).
- **보정의 대가**: 강한 축소는 과대평가는 없애지만 거의 사지 않아 이득도 0이 됩니다(5번). 약한 축소(8번)가 중간입니다.
- **망각**: 변화 후의 손실은 막지만, 모든 채널의 기록이 줄어들면 아무것도 사지 않는 상태에 갇힙니다(10번 후반 0.0).
  정리하신 원칙대로 망각은 **변화 감지(CUSUM)가 경보를 낼 때만** 적용하는 편이 맞아 보입니다.

**결론**: MU > 비용 규칙 자체는 맞습니다(1번). 하지만 총효용이 반드시 증가한다는 전제는
**추정 오차와 환경 변화가 있으면 깨집니다.** 그러니 총효용(Δ의 이동평균)은 최적화하지 않더라도 감시해야 합니다.

## 4. 직접 바꿔보기
`run()`의 인자를 바꿔 다른 조건을 시험할 수 있습니다.

- `cost`: 정보 1단위 비용. `n0`: 처음에 아는 채널별 이력 수 (작을수록 승자의 저주가 큼)
- `planA_noise`: 플랜 A의 잡음 (작을수록 플랜 A가 강함)
- `prior_n`, `z`: 축소 강도 (`estimator="shrink"`일 때)
- `forget`: 망각 계수 (1.0 = 안 잊음), `copy`, `drift`: 전제 위반 조건 켜기

In [ ]:
r = summarize("내 실험", seeds=20, estimator="naive", n0=50, cost=0.003, planA_noise=0.4)
{k: v for k, v in r.items()}